# 01 · Web Scraping — three Chinese-millennium-architecture datasets

**主题 / Theme:** 中国千禧年代地标建筑 (2000–2010, mainland China).

Goal — collect ≥250 elements from each of three different domains, all without OAuth or anti-bot evasion.

| # | Source | Domain | Modality | Retrieval philosophy |
|---|---|---|---|---|
| 1 | Wikidata SPARQL + zh.wiki REST | knowledge graph + encyclopedia | text | semantic structured query against the Wikidata RDF graph |
| 2 | Pixabay REST API | photo bank | image (metadata only) | keyword-driven full-text search |
| 3 | Bilibili search + danmaku | Chinese video social media | text (micro-comments) | distinctive Chinese-social-media surface form |

Each scraper writes raw JSONL into `data/raw/<source>/`. `preprocess.build_corpus()` then unifies them into a single Parquet file consumed by every later notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, logging
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(message)s')

from src.config import ensure_dirs, TARGET_ITEMS_PER_DATASET, THEME_NAME_ZH
ensure_dirs()
print('theme:', THEME_NAME_ZH)
print('target per dataset:', TARGET_ITEMS_PER_DATASET)

## 1.1 Wikidata SPARQL — semantic query for 2000–2010 Chinese buildings
We pose the question directly to the knowledge graph: *every architectural structure (Q811979) located in China (Q148) whose date of inception falls between 2000 and 2010*. For every QID we then fetch the zh.wiki REST summary, falling back to en.wiki when the Chinese article is missing.

In [ ]:
from src.scraping import scrape_wikidata_sparql
wikidata_path = scrape_wikidata_sparql()
wikidata_path

## 1.2 Pixabay — keyword-driven photo bank
Free CC0/PD photo API. Setup: drop your free key at the project root as `.pixabay_key` (gitignored).

In [ ]:
from src.scraping import scrape_pixabay
try:
    pixabay_path = scrape_pixabay()
except RuntimeError as exc:
    print('Pixabay key missing — see message:', exc)
    pixabay_path = None
pixabay_path

## 1.3 Bilibili — Chinese video search + danmaku (live-comment) extraction
We search the public web-search endpoint with our seed keywords, then hit the public danmaku XML stream for every BVID. Both endpoints are read-only and require no OAuth.

In [ ]:
from src.scraping import scrape_bilibili
bilibili_path = scrape_bilibili()
bilibili_path

## 1.4 OpenAI augmentation — synthesise 50 Chinese architecture reviews
Uses `gpt-4o-mini` with a Chinese system prompt asking for **structured JSON** output (title / city / year / typology / text). Drops a `.openai_key` file at the project root, or set `OPENAI_API_KEY`.

In [ ]:
from src.api import generate_synthetic_descriptions
import os
openai_path = generate_synthetic_descriptions(
    dry_run=os.environ.get('OPENAI_API_KEY') is None and not (ROOT/'.openai_key').exists()
)
openai_path

## 1.5 Build the unified corpus
Merge all four datasets into one Parquet (`data/processed/corpus.parquet`) — every later notebook reads from this single source of truth.

In [ ]:
from src.preprocess import build_corpus
corpus = build_corpus()
print('corpus shape:', corpus.shape)
corpus.groupby('source').size()

In [ ]:
corpus.head(3)[['id', 'source', 'title', 'text']]